# AISKG Framework v3.0.0 — Complete Unified Pipeline

This notebook executes the frozen manuscript-compatible Section 1 and Section 2 workflows, the nine-configuration ablation suite, publication figures, reproducibility audits, and deterministic release packaging.

[Open this notebook in Google Colab](https://colab.research.google.com/github/romenmeitei/AISKG_Framework/blob/main/notebooks/AISKG_Framework_v3_Complete_Pipeline.ipynb)

In [ ]:
#@title Run configuration
REPOSITORY_URL = "https://github.com/romenmeitei/AISKG_Framework.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
CONFIG_PATH = "configs/manuscript_frozen.yaml" #@param {type:"string"}
RUN_ID = "colab-publication-v3" #@param {type:"string"}
FORCE_RECLONE = False #@param {type:"boolean"}


In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

local_override = os.environ.get("AISKG_LOCAL_REPOSITORY")
if local_override:
    repository = pathlib.Path(local_override).expanduser().resolve()
else:
    repository = pathlib.Path("/content/AISKG_Framework")
    if FORCE_RECLONE and repository.exists():
        shutil.rmtree(repository)
    if not repository.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPOSITORY_URL, str(repository)], check=True)

os.chdir(repository)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-build-isolation"], check=True)
print(f"Repository: {repository}")
print(f"Python: {sys.version.split()[0]}")


In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    "run_pipeline.py",
    "--config", CONFIG_PATH,
    "--run-id", RUN_ID,
    "--clean",
]
print("Executing:", " ".join(command))
subprocess.run(command, check=True)


In [ ]:
from pathlib import Path
import pandas as pd
from aiskg.reproducibility import verify_run

run_dir = Path("outputs") / RUN_ID
verification = verify_run(run_dir)
audit = pd.read_csv(run_dir / "outputs" / "reproducibility_audit.csv")
ablation = pd.read_csv(run_dir / "outputs" / "extensions" / "ablation" / "ablation_summary.csv")

print((run_dir / "PIPELINE_SUCCESS.txt").read_text().strip())
print(f"Audit: {(audit.status == 'PASS').sum()}/{len(audit)} checks passed")
print("Verification:", verification)
display(ablation[[
    "label", "entity_f1", "relation_f1", "exact_triple_accuracy",
    "nodes", "edges", "modularity", "pathway_count"
]])
release_zip = run_dir / "AISKG_Framework_v3.0.0_Release.zip"
print("Release archive:", release_zip)


In [ ]:
# Download the deterministic release when running in Google Colab.
try:
    from google.colab import files
    files.download(str(release_zip))
except ImportError:
    print("Not running inside Google Colab; release remains at:", release_zip)
